# Problem 82 - Path Sum: Three Ways

> **NOTE:** This problem is a significantly more challenging version of **Problem 81**.

In the $5$ by $5$ matrix below, the minimal path sum from the top left to the bottom right, by moving left, right, up, and down, is indicated in **bold red** and is equal to $2297$.

$$
\begin{pmatrix}
\mathbf{131} & 673 & \mathbf{234} & \mathbf{103} & \mathbf{18} \\
\mathbf{201} & \mathbf{96} & \mathbf{342} & 965 & \mathbf{150} \\
630 & 803 & 746 & \mathbf{422} & \mathbf{111} \\
537 & 699 & 497 & \mathbf{121} & 956 \\
805 & 732 & 524 & \mathbf{37} & \mathbf{331}
\end{pmatrix}
$$

Find the minimal path sum from the top left to the bottom right by moving left, right, up, and down in `matrix.txt` (right click and "Save Link/Target As..."), a 31K text file containing an $80$ by $80$ matrix.

## Solution.

In [1]:
from tqdm import tqdm

In [2]:
with open('data_82.txt', 'r') as file:
    s = file.read()

data = [[int(x) for x in row.split(',') if x != ''] for row in s.split('\n') if len(row)>0]

In [3]:
# Save columns of matrix
n = len(data)
m = len(data[0])
matrix_columns = [[0] * n for _ in range(m)]

for j in range(m):
    for i in range(n):
        matrix_columns[j][i] = data[i][j]

In [24]:
def helper_dp(column1, column2, i):
    n = len(column1)
    dp = [[0]*n for _ in range(2)]

    dp[0][i] = column1[i]
    dp[1][i] = dp[0][i] + column2[i]
    
    for j in range(i-1, -1, -1):
        dp[0][j] = dp[0][j+1] + column1[j]
        dp[1][j] = min(dp[1][j+1], dp[0][j]) + column2[j]

    for j in range(i+1, n, 1):
        dp[0][j] = dp[0][j-1] + column1[j]
        dp[1][j] = min(dp[1][j-1], dp[0][j]) + column2[j]

    dp[1][i] = dp[0][i] + column2[i]

    return dp


def dp(columns, i):
    n = len(columns[0])
    m = len(columns)

    dp = [[float('inf')]*n for _ in range(m)] 

    dp[0], dp[1] = helper_dp(columns[0], columns[1], i)

    for j in range(2, m-1):
        ans = [0] * n
        dp[j] = [x+ y for (x,y) in zip(dp[j-1], columns[j])]

        for k in range(n):
            for l in range(k-1, -1, -1):
                dp[j][k] = min(dp[j][k], dp[j-1][l] + sum(columns[j][l:(k+1)]))
        
            for l in range(k+1, n, 1):
                dp[j][k] = min(dp[j][k], dp[j-1][l] + sum(columns[j][k:(l+1)]))

    
    last = []
    for k in range(n):
        dp[-1][k] = (dp[-2][k] + columns[-1][k])


    return min(dp[-1])

In [25]:
min(dp(matrix_columns, i) for i in tqdm(range(len(matrix_columns[0]))))

  0%|          | 0/80 [00:00<?, ?it/s]

100%|██████████| 80/80 [00:12<00:00,  6.49it/s]


260324

# Dijkstra

In [54]:
import heapq

def dijkstra(V, E, start):
    d = {v: float('inf') for v in V}
    d[start] = 0
    visited = set()
    pq = [(0, start)]

    while pq:
        du, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        for v, w in E[u].items():
            nd = du + w
            if nd < d[v]:
                d[v] = nd
                heapq.heappush(pq, (nd, v))

    return d

In [55]:
with open('data_82.txt', 'r') as file:
    s = file.read()

M = [[int(x) for x in row.split(',') if x != ''] for row in s.split('\n') if len(row)>0]

In [56]:
n = len(data)
m = len(data[0])

V = set([(x, y) for x in range(n) for y in range(m)])

def neighbourhood(v, n, m):
    x, y = v
    candidates = [(x-1, y), (x+1, y), (x, y+1)]
    return [(a, b) for a, b in candidates if 0 <= a < n and 0 <= b < m]


E = {}
for v in V:
    d = {}
    for u in neighbourhood(v, n, m):
        d[u] = M[u[0]][u[1]]
    E[v] = d

In [58]:
ans = float('inf')

for s in range(n):
    D = dijkstra(V, E, (s, 0))

    for e in range(n):
        ans = min(ans, D[(e, m-1)] + M[s][0])

print(ans)

260324
